# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 03: Video Downloading, Validation & Caching

**Goal**: Download and validate product complaint videos from the Amazon HLS (`.m3u8`) and direct MP4 URLs provided in the Com-VID dataset.

**Key Capabilities**:
1. **FFmpeg & HLS Handling**: Directly streams and transcodes HLS M3U8 playlists into standardized MP4 containers.
2. **Deduplication**: Saves videos by `video_url_hash`, automatically preventing repeated downloads of the 86 shared video URLs.
3. **Resume Support**: Validates existing videos on disk; skips re-downloading valid files.
4. **Pilot Mode**: Runs a small test of 5 videos first to verify connectivity and codec compatibility before full batch runs.
5. **Validation**: Inspects duration, resolution, FPS, and audio streams with `ffprobe`.

In [ ]:
# Environment & Kaggle Internet Verification
import os
import sys
import subprocess
import shutil
from pathlib import Path

# Ensure project root in sys.path
CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

# Verify FFmpeg availability
ffmpeg_bin = shutil.which('ffmpeg')
ffprobe_bin = shutil.which('ffprobe')
print(f'FFmpeg binary : {ffmpeg_bin}')
print(f'FFprobe binary: {ffprobe_bin}')
assert ffmpeg_bin and ffprobe_bin, 'FFmpeg and FFprobe must be installed on system!'

### 1. Configure Paths & Check Cleaned Dataset
Detects Kaggle vs local paths and loads `ComVID_cleaned.csv`.

In [ ]:
# Path Resolution
if Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv').exists():
    CLEANED_CSV = Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv')
elif Path('data/processed/ComVID_cleaned.csv').exists():
    CLEANED_CSV = Path('data/processed/ComVID_cleaned.csv')
else:
    CLEANED_CSV = Path('../data/processed/ComVID_cleaned.csv')

VIDEO_DIR = Path('data/videos')
REPORT_PATH = Path('reports/video_download_report.json')
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'Cleaned CSV : {CLEANED_CSV.resolve()}')
print(f'Video Output: {VIDEO_DIR.resolve()}')

### 2. Run Pilot Video Download (5 Videos)
We run `VideoDownloader` in pilot mode to download and validate 5 sample videos. In Kaggle, ensure 'Internet' is toggled ON in notebook settings.

In [ ]:
from src.vision.video_downloader import VideoDownloader
from src.vision.video_validator import VideoValidator

# Initialize Downloader in Pilot Mode (5 videos)
downloader = VideoDownloader(
    video_dir=VIDEO_DIR,
    report_path=REPORT_PATH,
    timeout_seconds=60,
    max_retries=3,
    pilot_mode=True,
    pilot_size=5
)

print('Starting Pilot Download (5 videos)...')
pilot_report = downloader.download_dataset(csv_path=CLEANED_CSV, limit=5)

print('\nPilot Download Summary:')
for k, v in pilot_report['summary'].items():
    print(f'  {k:20s}: {v}')

### 3. Inspect Downloaded Video Stream Metadata
Let's inspect the video files downloaded during the pilot run using `VideoValidator`.

In [ ]:
import glob

downloaded_files = sorted(list(VIDEO_DIR.glob('*.mp4')))
print(f'Total MP4 files present in {VIDEO_DIR}: {len(downloaded_files)}\n')

for vid in downloaded_files[:5]:
    info = VideoValidator.inspect(vid)
    print(f"File: {vid.name}")
    print(f"  Valid      : {info['is_valid']}")
    print(f"  Resolution : {info['width']}x{info['height']}")
    print(f"  FPS        : {info['fps']}")
    print(f"  Duration   : {info['duration_seconds']}s")
    print(f"  Has Audio  : {info['has_audio']} ({info.get('audio_codec')})")
    print(f"  File Size  : {info['file_size_bytes'] / (1024*1024):.2f} MB")
    print('-' * 40)

### 4. Full Dataset Batch Download (Optional / On-Demand)
To download the entire dataset, set `run_full_download = True`. Thanks to hash deduplication and resume checking, interrupted downloads can be restarted safely without re-downloading.

In [ ]:
run_full_download = False  # Set to True to trigger full batch download

if run_full_download:
    batch_downloader = VideoDownloader(
        video_dir=VIDEO_DIR,
        report_path=REPORT_PATH,
        timeout_seconds=60,
        max_retries=3,
        pilot_mode=False
    )
    print('Starting Full Dataset Download...')
    full_report = batch_downloader.download_dataset(csv_path=CLEANED_CSV)
    print('Full Download Summary:', full_report['summary'])
else:
    print('Skipping full download. Pilot verification complete.')

### Summary & Next Steps
- Pilot downloading, validation, and metadata extraction are working.
- Deduplication and resume capabilities protect against redundant network transfers.
- Proceed to **Notebook 04: Multimodal Preprocessing** for frame sampling and audio extraction.